In [1]:
 
#import pandas

import pandas as pd 


##1. Read all datasets with pandas 

customers = pd.read_csv("/Users/andreams/Downloads/files_mini_project/customers.csv")

orders = pd.read_csv("/Users/andreams/Downloads/files_mini_project/orders.csv")

products = pd.read_csv("/Users/andreams/Downloads/files_mini_project/products.csv")

clickstream = pd.read_csv("/Users/andreams/Downloads/files_mini_project/clickstream.csv")




print(customers.head())


   customer_id     customer_name                          email        city  \
0            1  William Jennings  william.jennings1@example.com      Dublin   
1            2      Diane Newman      diane.newman2@example.com  Manchester   
2            3     Samuel Wright     samuel.wright3@example.com      London   
3            4    Gillian Barnes    gillian.barnes4@example.com      Lisbon   
4            5     Nigel Edwards     nigel.edwards5@example.com     Glasgow   

  signup_date  
0  2025-01-29  
1  2025-11-06  
2  2024-08-12  
3  2025-09-15  
4  2025-11-23  


In [2]:
print(customers.shape)

(100, 5)


In [3]:
##2. Validate schema consistency. 

#column names check

print(customers.columns.tolist())
print(orders.columns.tolist())
print(clickstream.columns.tolist())
print(products.columns.tolist())    

['customer_id', 'customer_name', 'email', 'city', 'signup_date']
['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'payment_status']
['event_id', 'customer_id', 'event_type', 'page_url', 'event_timestamp', 'device_type']
['product_id', 'product_name', 'category', 'supplier_id', 'cost_price', 'selling_price']


In [4]:
##check data types

print(customers.dtypes)
print(orders.dtypes)
print(clickstream.dtypes)
print(products.dtypes)

customer_id       int64
customer_name    object
email            object
city             object
signup_date      object
dtype: object
order_id            int64
customer_id         int64
product_id          int64
order_date         object
quantity            int64
unit_price        float64
payment_status     object
dtype: object
event_id             int64
customer_id        float64
event_type          object
page_url            object
event_timestamp     object
device_type         object
dtype: object
product_id         int64
product_name      object
category          object
supplier_id        int64
cost_price       float64
selling_price    float64
dtype: object


In [5]:
##datatype corrections and Date conversions

customers["signup_date"] = pd.to_datetime(
    customers["signup_date"]
)

orders["order_date"] = pd.to_datetime(
    orders["order_date"]
)





# Fix customer_id type
clickstream["customer_id"] = (
    clickstream["customer_id"]
    .astype("Int64")
)




In [6]:
##check data types after corrections

print(customers.dtypes)
print(orders.dtypes)
print(clickstream.dtypes)
print(products.dtypes)

customer_id               int64
customer_name            object
email                    object
city                     object
signup_date      datetime64[ns]
dtype: object
order_id                   int64
customer_id                int64
product_id                 int64
order_date        datetime64[ns]
quantity                   int64
unit_price               float64
payment_status            object
dtype: object
event_id                    int64
customer_id                 Int64
event_type                 object
page_url                   object
event_timestamp    datetime64[ns]
device_type                object
dtype: object
product_id         int64
product_name      object
category          object
supplier_id        int64
cost_price       float64
selling_price    float64
dtype: object


In [7]:
###Detect duplicates

duplicates = customers.duplicated(subset="customer_id")

print(duplicates.sum())

0


In [8]:
duplicates = orders.duplicated(subset="order_id")

print(duplicates.sum())

0


In [9]:
duplicates = products.duplicated(subset="product_id")

print(duplicates.sum())

0


In [10]:
duplicates = clickstream.duplicated(subset="event_id")

print(duplicates.sum())

37


In [11]:
###find missing values 


customers.isnull().sum()

customer_id      0
customer_name    0
email            0
city             0
signup_date      0
dtype: int64

In [12]:
orders.isnull().sum()

order_id          0
customer_id       0
product_id        0
order_date        0
quantity          0
unit_price        0
payment_status    0
dtype: int64

In [13]:
clickstream.isnull().sum()

event_id              0
customer_id          60
event_type            0
page_url             83
event_timestamp    1309
device_type         424
dtype: int64

In [14]:
products.isnull().sum()

product_id       0
product_name     0
category         0
supplier_id      0
cost_price       0
selling_price    0
dtype: int64

In [15]:
##show rows with missing email

invalid_email = customers[
    customers["email"].isnull()
]

print(invalid_email)

Empty DataFrame
Columns: [customer_id, customer_name, email, city, signup_date]
Index: []


In [16]:
##find negative prices 

invalid_price  = orders[
    orders["unit_price"] < 0
]

print(invalid_price)


Empty DataFrame
Columns: [order_id, customer_id, product_id, order_date, quantity, unit_price, payment_status]
Index: []


In [17]:
##find negative quantity

invalid_quantity = orders[
    orders["quantity"] <= 0
]

print(invalid_quantity)


Empty DataFrame
Columns: [order_id, customer_id, product_id, order_date, quantity, unit_price, payment_status]
Index: []


In [18]:
#validate email format

invalid_email = customers[
    ~customers["email"].str.contains("@", na=False)
]

print(invalid_email)

Empty DataFrame
Columns: [customer_id, customer_name, email, city, signup_date]
Index: []


In [19]:
##check column names

expected_columns = [
    "customer_id",
    "customer_name",
    "email",
    "city",
    "signup_date"
]

###checking if columns match

set(expected_columns) == set(customers.columns)

True

In [20]:
###Q2. Data Cleaning & Transformation 

###1. Standardize city names. (if any)  

customers["city"] = (
    customers["city"]
    .str.strip()      # Remove extra spaces
    .str.title()      # Capitalize each word
)



In [ ]:
###2. Convert timestamps correctly.  (if any)

clickstream["event_timestamp"] = pd.to_datetime(
    clickstream["event_timestamp"],
    format="%d/%m/%Y %H:%M",
    errors="coerce"
)

In [41]:
##3. Handle missing values.  (if any)


###missing device_type

clickstream["device_type"] = clickstream["device_type"].fillna("Unknown")

In [36]:
##Handle missing page_url

# Fill missing page URLs 
clickstream["page_url"] = clickstream["page_url"].fillna(
    "Unknown"
)

In [39]:



###4. Remove corrupted rows.  

# Remove invalid timestamps
clickstream = clickstream.dropna(
    subset=["event_timestamp"]
)


In [40]:
# Remove duplicate rows
clickstream = clickstream.drop_duplicates()

In [43]:


###check null values 
clickstream.isnull().sum()

event_id            0
customer_id        22
event_type          0
page_url            0
event_timestamp     0
device_type         0
dtype: int64

In [46]:
####Standardize

clickstream["event_type"] = (
    clickstream["event_type"]
    .str.strip()  ###remove spaces 
    .str.lower()  ###low cap
)

###standarize event type 
clickstream["event_type"] = clickstream["event_type"].replace({
    "clik": "click",
    "pageview": "page_view",
    "page_view": "page_view",
    "add-to-cart": "add_to_cart",
    "add to cart": "add_to_cart",
    "log_out": "logout",
    "log out": "logout"
})

In [49]:
###Q3. Clickstream Analytics 


##1. Find the most visited pages.  

most_visited_pages = (
    clickstream["page_url"]
    .value_counts()
    .head(10)
)

print(most_visited_pages)



page_url
/products       79
/search         77
/product/105    71
/account        68
/cart           65
/home           64
/checkout       62
/orders         60
/help           58
/product/101    54
Name: count, dtype: int64


In [50]:
##2. Calculate session counts.  

# Sort the clickstream events by customer and time.
clickstream = clickstream.sort_values(
    ["customer_id", "event_timestamp"]
)

In [51]:

# Calculate the time difference between each event and the previous event
# for the same customer.

clickstream["time_diff"] = (
    clickstream
    .groupby("customer_id")["event_timestamp"]
    .diff()
)



In [52]:
### Create a column that identifies when a new session starts.

clickstream["new_session"] = (
    clickstream["time_diff"] > pd.Timedelta(minutes=30)
)

In [53]:
# Create a session ID for each group of events.

clickstream["session_id"] = (
    clickstream
    .groupby("customer_id")["new_session"]
    .cumsum()
)



In [54]:
### Count how many unique sessions each customer had.

sessions_per_customer = (
    clickstream
    .groupby("customer_id")["session_id"]
    .nunique()
)


In [55]:
# Add all customers' sessions together
# to get the total number of sessions.
session_count = sessions_per_customer.sum()


# Display the total session count
print(session_count)

669


In [56]:
## 3. Find bounce rate.  

# Count how many pages/events happened in each session
session_activity = (
    clickstream
    .groupby("session_id")
    .size()
)

# Count sessions with only one event (bounces)
bounces = (
    session_activity == 1
).sum()

# Count total sessions
total_sessions = session_activity.count()

# Calculate bounce rate percentage
bounce_rate = (
    bounces / total_sessions
) * 100

print(bounce_rate)


23.52941176470588


In [59]:
# Standardize device names (remove capitalization differences)
clickstream["device_type"] = (
    clickstream["device_type"]
    .str.strip()
    .str.lower()
)

print(clickstream["device_type"].unique())

['unknown' 'mobile' 'tablet' 'desktop']


In [60]:
##4. Find mobile vs desktop traffic percentage.  


device_traffic_percentage = (
    clickstream["device_type"]
    .value_counts(normalize=True)
    * 100
)

print(device_traffic_percentage)


device_type
mobile     35.311143
desktop    23.444284
tablet     21.562952
unknown    19.681621
Name: proportion, dtype: float64


In [65]:
##1. Export analytical dataset into:  CSV  and Parquet 


# Export customers dataset
customers.to_csv("/Users/andreams/Downloads/files_mini_project/analytical_datasets/customers_clean.csv", index=False)
customers.to_parquet("/Users/andreams/Downloads/files_mini_project/analytical_datasets/customers_clean.parquet", index=False)

In [67]:
# Export clickstream dataset
clickstream.to_csv("/Users/andreams/Downloads/files_mini_project/analytical_datasets/clickstream_clean.csv", index=False)
clickstream.to_parquet("/Users/andreams/Downloads/files_mini_project/analytical_datasets/clickstream_clean.parquet", index=False)

In [68]:
# Export orders dataset
orders.to_csv("/Users/andreams/Downloads/files_mini_project/analytical_datasets/orders_clean.csv", index=False)
orders.to_parquet("/Users/andreams/Downloads/files_mini_project/analytical_datasets/orders_clean.parquet", index=False)

In [69]:
# Export products dataset
products.to_csv("/Users/andreams/Downloads/files_mini_project/analytical_datasets/products_clean.csv", index=False)
products.to_parquet("/Users/andreams/Downloads/files_mini_project/analytical_datasets/products_clean.parquet", index=False)

In [71]:
##2. Compare storage sizes.  

import os

# Get file sizes in bytes
csv_size = os.path.getsize("/Users/andreams/Downloads/files_mini_project/analytical_datasets/clickstream_clean.csv")
parquet_size = os.path.getsize("/Users/andreams/Downloads/files_mini_project/analytical_datasets/clickstream_clean.parquet")

# Convert bytes to MB
csv_size_mb = csv_size / (1024 * 1024)
parquet_size_mb = parquet_size / (1024 * 1024)

print("CSV size:", round(csv_size_mb, 2), "MB")
print("Parquet size:", round(parquet_size_mb, 2), "MB")

CSV size: 0.05 MB
Parquet size: 0.02 MB


In [72]:
##3. Compare read performance.  

import time
import pandas as pd

start = time.time()
pd.read_csv("/Users/andreams/Downloads/files_mini_project/analytical_datasets/clickstream_clean.csv")
csv_time = time.time() - start


start = time.time()
pd.read_parquet("/Users/andreams/Downloads/files_mini_project/analytical_datasets/clickstream_clean.parquet")
parquet_time = time.time() - start


print("CSV read time:", csv_time)
print("Parquet read time:", parquet_time)


CSV read time: 0.01240086555480957
Parquet read time: 3.2566821575164795


In [73]:
import pandas as pd
import time

# CSV path
csv_file = "/Users/andreams/Downloads/files_mini_project/analytical_datasets/clickstream_clean.csv"

# Parquet path
parquet_file = "/Users/andreams/Downloads/files_mini_project/analytical_datasets/clickstream_clean.parquet"


# Measure CSV reading time
start = time.time()

pd.read_csv(csv_file)

csv_time = time.time() - start


# Measure Parquet reading time
start = time.time()

pd.read_parquet(parquet_file)

parquet_time = time.time() - start


print("CSV read time:", csv_time, "seconds")
print("Parquet read time:", parquet_time, "seconds")

CSV read time: 0.007135152816772461 seconds
Parquet read time: 0.0039250850677490234 seconds


In [76]:
import os

print(os.getcwd())

/Users/andreams
